In [3]:
!pip install nltk
!pip install tflearn
!pip install tensorflow

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for tflearn: filename=tflearn-0.5.0-py3-none-any.whl size=127331 sha256=cbbb2b5deff750096eafd251affadf8d90b3217589525e355e81b358b6e3efc6
  Stored in directory: /root/.cache/pip/wheels/fe/d0/a7/a3d3502219f7cd20ab21645569b9929fb1b47bcad206cf1f79
Successfully built tflearn


In [11]:
import nltk
from nltk.stem.lancaster import LancasterStemmer
stemmer = LancasterStemmer()

# Dynamic patch to fix tflearn compatibility with modern TensorFlow and Pillow
import tensorflow as tf
import sys
import PIL.Image
import os

# Ensure TensorFlow runs in compatibility mode
import tensorflow.compat.v1 as tf
tf.disable_v2_behavior()

# Patch missing 'is_sequence' attribute inside tensorflow's internal nest utility
try:
    import tensorflow.python.util.nest as tf_nest
    if not hasattr(tf_nest, 'is_sequence'):
        import collections.abc
        def is_sequence(seq):
            return isinstance(seq, collections.abc.Sequence) and not isinstance(seq, (str, bytes))
        tf_nest.is_sequence = is_sequence
except ImportError:
    pass

# Patch missing 'ANTIALIAS' in Pillow (removed in modern Pillow versions)
if not hasattr(PIL.Image, 'ANTIALIAS'):
    PIL.Image.ANTIALIAS = PIL.Image.Resampling.LANCZOS

import numpy as np
import tflearn as tl
import random, pickle, json

# Download necessary NLTK resources
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

with open("/content/intents.json") as f:
    data = json.load(f)

try:
    with open("data.pickle", "rb") as m:
        words, labels, training, output = pickle.load(m)
except:
    words = []
    labels = []
    docs_x = []
    docs_y = []

    for intent in data["intents"]:
        for pattern in intent["patterns"]:
            wrds = nltk.word_tokenize(pattern)
            words.extend(wrds)
            docs_x.append(wrds)
            docs_y.append(intent["tag"])
        if intent["tag"] not in labels:
            labels.append(intent["tag"])
    words = [stemmer.stem(w.lower()) for w in words if w!="?"]
    words = sorted(list(set(words)))

    labels = sorted(labels)

    training = []
    output = []

    out_empty = [0 for _ in range(len(labels))]

    for x, doc in enumerate(docs_x):
        bag = []
        wrds = [stemmer.stem(w.lower()) for w in doc]

        for w in words:
            if w in wrds:
                bag.append(1)
            else:
                bag.append(0)

        output_row = out_empty[:]
        output_row[labels.index(docs_y[x])] = 1

        training.append(bag)
        output.append(output_row)

    training = np.array(training)
    output = np.array(output)

    with open("data.pickle", "wb") as m:
        pickle.dump((words, labels, training, output), m)

tf.reset_default_graph()

net = tl.input_data(shape=[None, len(training[0])])
net = tl.fully_connected(net, 8)
net = tl.fully_connected(net, 8)
net = tl.fully_connected(net, len(output[0]), activation="softmax")
net = tl.regression(net)

model = tl.DNN(net)

# Check if model files exist on disk before calling load to prevent closing the session
if os.path.exists("model.tflearn.meta"):
    model.load("model.tflearn")
else:
    model.fit(training, output, n_epoch=1000, batch_size=8, show_metric=True)
    model.save("model.tflearn")

def bag_of_words(s, words):
    bag = [0 for _ in range(len(words))]

    s_words = nltk.word_tokenize(s)
    s_words = [stemmer.stem(word.lower()) for word in s_words]

    for se in s_words:
        for i, w in enumerate(words):
            if w == se:
                bag[i] = 1

    return np.array(bag)

def chat():
    print("Start talking with the bot (type quit to stop)!")
    while True:
        inp = input("You: ")
        if inp.lower() == "quit":
            break

        results = model.predict([bag_of_words(inp, words)])[0]
        results_index = np.argmax(results)
        tag = labels[results_index]

        if results[results_index] > 0.7:
            for tg in data["intents"]:
                if tg['tag'] == tag:
                    responses = tg['responses']
            print(random.choice(responses))
        else:
            print("I didn't get that, try again.")
chat()

Start talking with the bot (type quit to stop)!
You: what do you sell
Cookies are on the menu!
You: how much
I didn't get that, try again.
You: how much?
I didn't get that, try again.
You: how much a cookies?
Hello!
You: can you do math?
I didn't get that, try again.
You: quir
I didn't get that, try again.
You: quit
